In [7]:
import pandas as pd, numpy as np

%load_ext autoreload
%autoreload 2

!date

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Mon Jul 22 12:48:00 PDT 2024


# HCES Data Extraction
We have HCES data to inform rice consumption (of PSD-distributed and non-PSD rice) by women and birthing people of reproductive age (WBPRA) and U5 children in India. We need to do further investigation as to what percentage of PSD-distributed rice has been fortified with iron and folate.

HCES data and documentation is saved here: https://uwnetid.sharepoint.com/:f:/r/sites/ihme_simulation_science_team/Shared%20Documents/Research/LSFF/07_Data/HCES_22_data_files?csf=1&web=1&e=wmxQSD

Item codes for 'Rice' include: 101, 102, and 061. See Section 5.1 of the above documentation for more context.
- Item code 101 signifies rice procured through PDS, using ration card.
- Item code 061 signifies rice procured through PDS, free of charge.
- Item coe 102 signifies rice procured/consumed from other sources: presumably this is unfortified rice

TODO: What about rice products? (e.g. muri)

We also will need to approximate DHS wealth quintiles in the HCES data, by using similar variables as is used by the DHS to calculate wealth quintiles in India (see DHS wealth quintile documentation here: https://dhsprogram.com/programming/wealth%20index/India%20DHS%202015-16/India%202015-16%20sps.txt)

In [41]:
df = pd.read_fwf('data/hces22_lvl_04.TXT', header=None, widths=[38,1,2,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,4,15])

In [42]:
df

,0,1,2,3,4,5,6,7,8,9,...,13,14,15,16,17,18,19,20,21,22
0,HCES2022616751181822223011 118101 311,F,4,1,1.0,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,5,134372.0
1,HCES20226660710808514232010108105 204,F,4,1,NaN,1.0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,2000,169359.0
2,HCES2022398952323220812071 132214 101,F,4,1,1.0,1.0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,1200,612500.0
3,HCES2022695011202021013018 120153 301,F,4,1,1.0,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,160,84913.0
4,HCES2022346562333343232011 233101 304,F,4,2,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,1000,27434.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
261741,HCES20223951022828313220210228101 313,F,4,2,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2,0,73515.0
261742,HCES20223951022828313220210228101 314,F,4,1,1.0,NaN,NaN,1.0,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2,0,73515.0
261743,HCES20223951022828313220210228101 315,F,4,2,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2,0,73515.0
261744,HCES20223951022828313220210228101 316,F,4,2,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2,0,73515.0


In [44]:
df[4].value_counts()
# Rice - Q4.1.2

1.0    163929
Name: 4, dtype: int64

In [45]:
df[3].value_counts()

1    177001
2     84745
Name: 3, dtype: int64

In [46]:
df2 = pd.read_fwf('data/hces22_lvl_05.TXT', header=None, widths = [38,1,2,3,10,8,10,8,1,15])
df2

,0,1,2,3,4,5,6,7,8,9
0,HCES2022655561010131113011 101202 201,F,5,102,NaN,NaN,40,1200.0,1.0,35560.0
1,HCES2022655561010131113011 101202 201,F,5,143,NaN,NaN,5,600.0,1.0,35560.0
2,HCES2022655561010131113011 101202 201,F,5,160,NaN,NaN,14,420.0,1.0,35560.0
3,HCES2022655561010131113011 101202 201,F,5,163,NaN,NaN,3,150.0,1.0,35560.0
4,HCES2022655561010131113011 101202 201,F,5,166,NaN,NaN,NaN,60.0,1.0,35560.0
...,...,...,...,...,...,...,...,...,...,...
12056834,HCES20223492023737102120110201201 313,F,5,249,NaN,NaN,0.100,70.0,NaN,938.0
12056835,HCES20223492023737102120110201201 313,F,5,199,NaN,NaN,NaN,350.0,NaN,938.0
12056836,HCES20223492023737102120110201201 313,F,5,189,NaN,NaN,0.750,135.0,NaN,938.0
12056837,HCES20223492023737102120110201201 313,F,5,269,NaN,NaN,1100.000,210.0,NaN,938.0


In [47]:
df2[3].value_counts()

179    257820
219    257253
129    257178
189    257105
269    256967
        ...  
68         31
57         26
60          4
66          3
58          1
Name: 3, Length: 164, dtype: int64

In [109]:
nonpds_df = df2[df2[3] == 102]
nonpds_df
# non-PDS rice

,0,1,2,3,4,5,6,7,8,9
0,HCES2022655561010131113011 101202 201,F,5,102,NaN,NaN,40,1200.0,1.0,35560.0
26,HCES2022655561010131113011 101202 301,F,5,102,40,950,40,950.0,2.0,30331.0
53,HCES2022655561010131113011 101202 302,F,5,102,30,900,36,1020.0,3.0,30331.0
80,HCES2022655561010131113011 101202 303,F,5,102,NaN,NaN,25,500.0,1.0,30331.0
106,HCES2022655561010131113011 101202 304,F,5,102,20,600,20,600.0,2.0,30331.0
...,...,...,...,...,...,...,...,...,...,...
9314415,HCES20223951022828313220210228101 314,F,5,102,NaN,NaN,15.00,750.0,1.0,73515.0
9314452,HCES20223951022828313220210228101 315,F,5,102,NaN,NaN,25.00,1300.0,1.0,73515.0
9314493,HCES20223951022828313220210228101 316,F,5,102,NaN,NaN,25.00,1300.0,1.0,73515.0
9314532,HCES20223951022828313220210228101 317,F,5,102,NaN,NaN,25.00,1300.0,1.0,73515.0


In [103]:
pds_df = df2[(df2[3] == 101) | (df2[3] == 61)]
pds_df
# PDS rice

,0,1,2,3,4,5,6,7,8,9
133,HCES2022655561010131113011 101202 305,F,5,101,NaN,NaN,20,240.0,1.0,30331.0
354,HCES2022655561010131113011 101202 314,F,5,101,NaN,NaN,10,130.0,1.0,30331.0
448,HCES2022655531010131213011 201212 301,F,5,101,NaN,NaN,50.00,250.0,1.0,23213.0
465,HCES2022655531010131213011 201212 302,F,5,101,NaN,NaN,50.000,250.0,1.0,23213.0
489,HCES2022655531010131213011 201212 303,F,5,101,NaN,NaN,40.000,120.0,1.0,23213.0
...,...,...,...,...,...,...,...,...,...,...
9314299,HCES20223951022828313220210228101 311,F,5,61,NaN,NaN,10.00,NaN,1.0,73515.0
9314340,HCES20223951022828313220210228101 312,F,5,61,NaN,NaN,10.00,NaN,1.0,73515.0
9314414,HCES20223951022828313220210228101 314,F,5,61,NaN,NaN,10.00,NaN,1.0,73515.0
9314531,HCES20223951022828313220210228101 317,F,5,61,NaN,NaN,10.00,NaN,1.0,73515.0


In [104]:
# Columns 6 and 7 denote the quantity of total consumption (in kg), with column 6 being the integer and column 7 
# the fractional part? (See section 3.3.3.3 in HCES Volume I documentation) - So should I concatenate these two columns
# together? 
pds_df[6].describe()

count     179216
unique      1357
top           20
freq       14994
Name: 6, dtype: object

In [57]:
pds_df[7].value_counts().sort_values()
# All of these values should be max 3 digits - that one 4 digit value is probably an error, I think it's fine

928.0        1
470.0        1
408.0        1
1620.0       1
193.0        1
          ... 
10.0      3766
60.0      4060
20.0      4655
15.0      4765
30.0      5194
Name: 7, Length: 485, dtype: int64

In [110]:
# Let's rename the variables in these dfs so they are easier to deal with! 
pds_df = pds_df.rename({0: 'common_id', 6: 'pds_total_consumption_int', 7: 'pds_total_consumption_decimal'}, axis=1)
nonpds_df = nonpds_df.rename({0: 'common_id', 6: 'nonpds_total_consumption_int', 7: 'nonpds_total_consumption_decimal'}, axis=1)

In [106]:
pds_df

,common_id,1,2,3,4,5,pds_total_consumption_int,pds_total_consumption_decimal,8,9
133,HCES2022655561010131113011 101202 305,F,5,101,NaN,NaN,20,240.0,1.0,30331.0
354,HCES2022655561010131113011 101202 314,F,5,101,NaN,NaN,10,130.0,1.0,30331.0
448,HCES2022655531010131213011 201212 301,F,5,101,NaN,NaN,50.00,250.0,1.0,23213.0
465,HCES2022655531010131213011 201212 302,F,5,101,NaN,NaN,50.000,250.0,1.0,23213.0
489,HCES2022655531010131213011 201212 303,F,5,101,NaN,NaN,40.000,120.0,1.0,23213.0
...,...,...,...,...,...,...,...,...,...,...
9314299,HCES20223951022828313220210228101 311,F,5,61,NaN,NaN,10.00,NaN,1.0,73515.0
9314340,HCES20223951022828313220210228101 312,F,5,61,NaN,NaN,10.00,NaN,1.0,73515.0
9314414,HCES20223951022828313220210228101 314,F,5,61,NaN,NaN,10.00,NaN,1.0,73515.0
9314531,HCES20223951022828313220210228101 317,F,5,61,NaN,NaN,10.00,NaN,1.0,73515.0


Now we need to use these dfs to find rice consumption amounts in our populations of interest: WBPRA and U5 children. We 
will look to household characteristics to determine these numbers.

Info we have: 
- Item 5.5: household size (Level 03, Question 2.1) 

In [58]:
dfh = pd.read_fwf('data/hces22_lvl_03.TXT', header=None, widths = [38,1,2,2,1,3,5,1,1,1,1,1,1,1,1,1,9,1,1,1,1,1,2,1,2,3,1,2,1,1,1,1,2,15])
dfh

,0,1,2,3,4,5,6,7,8,9,...,24,25,26,27,28,29,30,31,32,33
0,HCES2022655561010131113011 101202 201,H,3,5,1,332.0,68200.0,1.0,2.0,NaN,...,2,0.0,1,2.0,3,2.0,2,2,0,35560.0
1,HCES2022655561010131113011 101202 301,H,3,6,1,931.0,42909.0,3.0,NaN,NaN,...,2,0.0,1,2.0,2,2.0,2,2,0,30331.0
2,HCES2022655561010131113011 101202 302,H,3,8,1,833.0,49211.0,1.0,2.0,NaN,...,2,0.0,1,2.0,3,2.0,2,2,0,30331.0
3,HCES2022655561010131113011 101202 303,H,3,4,1,142.0,47713.0,1.0,2.0,NaN,...,2,0.0,1,2.0,0,2.0,2,2,0,30331.0
4,HCES2022655561010131113011 101202 304,H,3,4,1,833.0,49211.0,1.0,2.0,NaN,...,2,0.0,1,2.0,3,2.0,2,2,0,30331.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
261741,HCES20223951022828313220210228101 314,H,3,2,1,832.0,49219.0,1.0,NaN,NaN,...,11,10.0,1,1.0,4,NaN,1,2,0,73515.0
261742,HCES20223951022828313220210228101 315,H,3,4,1,216.0,85500.0,2.0,NaN,NaN,...,2,0.0,1,1.0,0,NaN,1,2,0,73515.0
261743,HCES20223951022828313220210228101 316,H,3,3,1,411.0,84119.0,2.0,NaN,NaN,...,2,0.0,1,1.0,4,NaN,1,2,0,73515.0
261744,HCES20223951022828313220210228101 317,H,3,5,1,112.0,47711.0,1.0,NaN,NaN,...,2,0.0,1,1.0,4,NaN,1,2,0,73515.0


In [60]:
dfh[3].value_counts()
# This should be household size! The fact that there are a couple very large households (e.g. 20+ members) is a little 
# surprising - maybe we can investigate to confirm these are some kind of GQ. 

# Although not sure we need household size, pausing this for now.

4     64290
5     46960
3     42596
2     31243
6     27890
1     18181
7     13421
8      7183
9      4008
10     2573
11     1390
12      790
13      446
14      266
15      164
16      138
17       68
18       44
20       30
19       25
21       13
22        9
24        5
27        3
28        2
23        2
25        2
30        1
37        1
31        1
29        1
Name: 3, dtype: int64

Let's see what we have for household member characteristics (Section 3: Details of the household members): 
- Column 4: gender (1 - male, 2 - female, 3 - transgender (hijras, eunuchs)) 
- Column 5: age (years)

This information is saved in the Level 02 file.

In [65]:
dfh = pd.read_fwf('data/hces22_lvl_02.TXT', header=None, widths = [38,1,2,2,1,1,3,1,2,2,1,2,1,2,2,2,2,2,2,1,1,15])
dfh

,0,1,2,3,4,5,6,7,8,9,...,12,13,14,15,16,17,18,19,20,21
0,HCES2022655561010131113011 101202 201,H,2,1,1,1,48,2,6,12.0,...,2.0,0.0,0.0,0.0,0.0,58.0,11,3,5,560.0
1,HCES2022655561010131113011 101202 201,H,2,2,2,2,46,2,1,NaN,...,2.0,NaN,NaN,NaN,NaN,60.0,11,3,5,560.0
2,HCES2022655561010131113011 101202 201,H,2,3,5,1,24,1,13,18.0,...,2.0,NaN,NaN,NaN,NaN,58.0,11,3,5,560.0
3,HCES2022655561010131113011 101202 201,H,2,4,5,1,18,1,7,13.0,...,2.0,NaN,NaN,NaN,NaN,56.0,11,3,5,560.0
4,HCES2022655561010131113011 101202 201,H,2,5,5,2,21,1,12,17.0,...,2.0,NaN,NaN,NaN,NaN,54.0,11,3,5,560.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1127034,HCES20223492023737102120110201201 313,H,2,1,1,1,40,2,12,17.0,...,2.0,0.0,0.0,0.0,5.0,55.0,11,9,3,8.0
1127035,HCES20223492023737102120110201201 313,H,2,2,2,2,38,2,6,12.0,...,2.0,0.0,0.0,0.0,0.0,60.0,11,9,3,8.0
1127036,HCES20223492023737102120110201201 313,H,2,3,5,1,12,1,4,7.0,...,2.0,0.0,0.0,0.0,0.0,60.0,11,9,3,8.0
1127037,HCES20223492023737102120110201201 313,H,2,4,5,2,7,1,3,2.0,...,2.0,0.0,0.0,0.0,0.0,60.0,11,9,3,8.0


In [68]:
dfh[5].value_counts()
# Gender 

1    574615
2    552033
3       391
Name: 5, dtype: int64

In [69]:
dfh[6].value_counts()
# Age
# WHO considers women 15-49 years old to be of reproductive age

30     32144
35     31767
40     31372
45     30790
25     26620
       ...  
108        4
115        2
120        1
117        1
106        1
Name: 6, Length: 113, dtype: int64

In [87]:
wra_df = dfh[(dfh[5] == 2) & ((dfh[6] >= 15) & (dfh[6] <= 49))] 
wra_df

,0,1,2,3,4,5,6,7,8,9,...,12,13,14,15,16,17,18,19,20,21
1,HCES2022655561010131113011 101202 201,H,2,2,2,2,46,2,1,NaN,...,2.0,NaN,NaN,NaN,NaN,60.0,11,3,5,560.0
4,HCES2022655561010131113011 101202 201,H,2,5,5,2,21,1,12,17.0,...,2.0,NaN,NaN,NaN,NaN,54.0,11,3,5,560.0
6,HCES2022655561010131113011 101202 301,H,2,2,2,2,45,2,1,NaN,...,2.0,NaN,NaN,NaN,NaN,60.0,11,3,0,331.0
9,HCES2022655561010131113011 101202 301,H,2,5,5,2,19,1,7,13.0,...,2.0,NaN,NaN,NaN,NaN,60.0,11,3,0,331.0
13,HCES2022655561010131113011 101202 302,H,2,3,4,2,33,2,1,NaN,...,2.0,NaN,NaN,NaN,NaN,58.0,11,3,0,331.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1127006,HCES20223492023737102120110201201 307,H,2,2,2,2,37,2,7,14.0,...,2.0,0.0,0.0,0.0,0.0,60.0,11,9,3,8.0
1127013,HCES20223492023737102120110201201 308,H,2,5,5,2,28,1,13,17.0,...,2.0,0.0,0.0,0.0,2.0,58.0,11,9,3,8.0
1127022,HCES20223492023737102120110201201 310,H,2,3,4,2,28,2,7,14.0,...,2.0,0.0,0.0,0.0,0.0,60.0,11,9,3,8.0
1127031,HCES20223492023737102120110201201 312,H,2,2,2,2,45,2,6,12.0,...,2.0,0.0,0.0,0.0,3.0,57.0,11,9,3,8.0


In [88]:
# Let's rename the columns so they are easier to deal with! 
wra_df = wra_df.rename({0: 'common_id', 5: 'gender', 6: 'age'}, axis=1)
wra_df

,common_id,1,2,3,4,gender,age,7,8,9,...,12,13,14,15,16,17,18,19,20,21
1,HCES2022655561010131113011 101202 201,H,2,2,2,2,46,2,1,NaN,...,2.0,NaN,NaN,NaN,NaN,60.0,11,3,5,560.0
4,HCES2022655561010131113011 101202 201,H,2,5,5,2,21,1,12,17.0,...,2.0,NaN,NaN,NaN,NaN,54.0,11,3,5,560.0
6,HCES2022655561010131113011 101202 301,H,2,2,2,2,45,2,1,NaN,...,2.0,NaN,NaN,NaN,NaN,60.0,11,3,0,331.0
9,HCES2022655561010131113011 101202 301,H,2,5,5,2,19,1,7,13.0,...,2.0,NaN,NaN,NaN,NaN,60.0,11,3,0,331.0
13,HCES2022655561010131113011 101202 302,H,2,3,4,2,33,2,1,NaN,...,2.0,NaN,NaN,NaN,NaN,58.0,11,3,0,331.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1127006,HCES20223492023737102120110201201 307,H,2,2,2,2,37,2,7,14.0,...,2.0,0.0,0.0,0.0,0.0,60.0,11,9,3,8.0
1127013,HCES20223492023737102120110201201 308,H,2,5,5,2,28,1,13,17.0,...,2.0,0.0,0.0,0.0,2.0,58.0,11,9,3,8.0
1127022,HCES20223492023737102120110201201 310,H,2,3,4,2,28,2,7,14.0,...,2.0,0.0,0.0,0.0,0.0,60.0,11,9,3,8.0
1127031,HCES20223492023737102120110201201 312,H,2,2,2,2,45,2,6,12.0,...,2.0,0.0,0.0,0.0,3.0,57.0,11,9,3,8.0


In [89]:
wra_df.age.value_counts().sort_values(ascending=False)

30    15849
40    14885
35    14779
45    14364
25    13135
28    12756
32    12130
18    11988
22    11978
20    11608
38    11528
15    10250
16     9918
42     9829
26     9819
24     9576
23     9276
17     9129
48     8496
19     8487
21     8119
27     8112
36     7542
34     6255
33     6176
29     6042
37     5530
46     5093
43     5028
39     4684
47     4510
31     4430
44     3963
41     3386
49     3348
Name: age, dtype: int64

In [94]:
u5_df = dfh[dfh[6] < 5] 
u5_df

# TODO: double-check that U5 children is not inclusive to 5 year olds

,0,1,2,3,4,5,6,7,8,9,...,12,13,14,15,16,17,18,19,20,21
18,HCES2022655561010131113011 101202 302,H,2,8,6,1,3,1,1,NaN,...,2.0,NaN,NaN,NaN,NaN,60.0,11,3,0,331.0
22,HCES2022655561010131113011 101202 303,H,2,4,5,1,4,1,3,1.0,...,2.0,NaN,NaN,NaN,NaN,60.0,11,3,0,331.0
26,HCES2022655561010131113011 101202 304,H,2,4,6,2,4,1,1,NaN,...,2.0,NaN,NaN,NaN,NaN,60.0,11,3,0,331.0
36,HCES2022655561010131113011 101202 306,H,2,5,6,1,4,1,1,NaN,...,2.0,NaN,NaN,NaN,NaN,60.0,11,3,0,331.0
50,HCES2022655561010131113011 101202 309,H,2,4,5,2,3,1,1,NaN,...,2.0,NaN,NaN,NaN,NaN,60.0,11,3,0,331.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1126897,HCES20223492723737101120110101201 104,H,2,5,6,1,2,1,1,NaN,...,3.0,0.0,NaN,NaN,NaN,90.0,11,3,9,54.0
1126903,HCES20223492723737101120110101201 105,H,2,4,6,1,1,1,1,NaN,...,3.0,NaN,NaN,NaN,NaN,90.0,11,3,9,54.0
1126942,HCES20223492723737101120110101201 302,H,2,3,5,1,4,1,1,NaN,...,3.0,24.0,0.0,NaN,NaN,66.0,11,2,3,85.0
1126969,HCES20223492023737102120110201201 203,H,2,6,6,1,2,1,1,NaN,...,2.0,0.0,0.0,0.0,0.0,60.0,11,8,4,8.0


In [95]:
u5_df = u5_df.rename({0: 'common_id', 5: 'gender', 6: 'age'}, axis=1)
u5_df

,common_id,1,2,3,4,gender,age,7,8,9,...,12,13,14,15,16,17,18,19,20,21
18,HCES2022655561010131113011 101202 302,H,2,8,6,1,3,1,1,NaN,...,2.0,NaN,NaN,NaN,NaN,60.0,11,3,0,331.0
22,HCES2022655561010131113011 101202 303,H,2,4,5,1,4,1,3,1.0,...,2.0,NaN,NaN,NaN,NaN,60.0,11,3,0,331.0
26,HCES2022655561010131113011 101202 304,H,2,4,6,2,4,1,1,NaN,...,2.0,NaN,NaN,NaN,NaN,60.0,11,3,0,331.0
36,HCES2022655561010131113011 101202 306,H,2,5,6,1,4,1,1,NaN,...,2.0,NaN,NaN,NaN,NaN,60.0,11,3,0,331.0
50,HCES2022655561010131113011 101202 309,H,2,4,5,2,3,1,1,NaN,...,2.0,NaN,NaN,NaN,NaN,60.0,11,3,0,331.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1126897,HCES20223492723737101120110101201 104,H,2,5,6,1,2,1,1,NaN,...,3.0,0.0,NaN,NaN,NaN,90.0,11,3,9,54.0
1126903,HCES20223492723737101120110101201 105,H,2,4,6,1,1,1,1,NaN,...,3.0,NaN,NaN,NaN,NaN,90.0,11,3,9,54.0
1126942,HCES20223492723737101120110101201 302,H,2,3,5,1,4,1,1,NaN,...,3.0,24.0,0.0,NaN,NaN,66.0,11,2,3,85.0
1126969,HCES20223492023737102120110201201 203,H,2,6,6,1,2,1,1,NaN,...,2.0,0.0,0.0,0.0,0.0,60.0,11,8,4,8.0


Now, let's merge together wra_df, pds_df, and nonpds_df, and separately u5_df, pds_df, and nonpds_df so we can actually
calculate consumption amounts of PDS and non-PDS rice! 

In [111]:
wra_rice_df = pd.merge(wra_df, pds_df, on='common_id', how='inner')
wra_rice_df = pd.merge(wra_rice_df, nonpds_df, on='common_id', how='inner')
wra_rice_df

,common_id,1_x,2_x,3_x,4_x,gender,age,7,8_x,9_x,...,9_y,1,2,3,4,5_y,nonpds_total_consumption_int,nonpds_total_consumption_decimal,8,9
0,HCES2022655561010131113011 101202 305,H,2,2,2,2,46,2,1,NaN,...,30331.0,F,5,102,10,300,10,300.0,2.0,30331.0
1,HCES2022655561010131113011 101202 305,H,2,5,5,2,24,1,7,13.0,...,30331.0,F,5,102,10,300,10,300.0,2.0,30331.0
2,HCES2022655561010131113011 101202 314,H,2,2,2,2,44,2,1,NaN,...,30331.0,F,5,102,30,900,30,900.0,2.0,30331.0
3,HCES2022655531010131213011 201212 302,H,2,4,4,2,33,2,3,4.0,...,23213.0,F,5,102,NaN,NaN,40.000,800.0,1.0,23213.0
4,HCES2022655531010131213011 201212 302,H,2,9,5,2,23,1,5,8.0,...,23213.0,F,5,102,NaN,NaN,40.000,800.0,1.0,23213.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
151038,HCES20223492023737102120110201201 307,H,2,2,2,2,37,2,7,14.0,...,938.0,F,5,102,NaN,NaN,10,400.0,1.0,938.0
151039,HCES20223492023737102120110201201 308,H,2,5,5,2,28,1,13,17.0,...,938.0,F,5,102,NaN,NaN,10,400.0,1.0,938.0
151040,HCES20223492023737102120110201201 310,H,2,3,4,2,28,2,7,14.0,...,938.0,F,5,102,NaN,NaN,10,400.0,1.0,938.0
151041,HCES20223492023737102120110201201 312,H,2,2,2,2,45,2,6,12.0,...,938.0,F,5,102,NaN,NaN,10,400.0,1.0,938.0


In [112]:
wra_rice_df.columns

Index([                       'common_id',                              '1_x',
                                    '2_x',                              '3_x',
                                    '4_x',                           'gender',
                                    'age',                                  7,
                                    '8_x',                              '9_x',
                                       10,                                 11,
                                       12,                                 13,
                                       14,                                 15,
                                       16,                                 17,
                                       18,                                 19,
                                       20,                                 21,
                                    '1_y',                              '2_y',
                                    '3_y',          

In [114]:
wra_rice_df['pds_total_consumption_int'] = wra_rice_df['pds_total_consumption_int'].astype(str)
wra_rice_df['nonpds_total_consumption_int'] = wra_rice_df['nonpds_total_consumption_int'].astype(str)
wra_rice_df['pds_total_consumption_decimal'] = wra_rice_df['pds_total_consumption_decimal'].astype(str)
wra_rice_df['nonpds_total_consumption_decimal'] = wra_rice_df['nonpds_total_consumption_decimal'].astype(str)

wra_rice_df['pds_total_consumption_decimal'] = wra_rice_df.pds_total_consumption_decimal.str.replace('.0', '')
wra_rice_df['nonpds_total_consumption_decimal'] = wra_rice_df.nonpds_total_consumption_decimal.str.replace('.0', '')

wra_rice_df['pds_total_consumption'] = wra_rice_df['pds_total_consumption_decimal'] + '.' + wra_rice_df['pds_total_consumption_decimal']
wra_rice_df['nonpds_total_consumption'] = wra_rice_df['nonpds_total_consumption_decimal'] + '.' + wra_rice_df['nonpds_total_consumption_decimal']

wra_rice_df['pds_total_consumption'] = pd.to_numeric(wra_rice_df['pds_total_consumption'], errors='coerce')
wra_rice_df['nonpds_total_consumption'] = pd.to_numeric(wra_rice_df['nonpds_total_consumption'], errors='coerce')

/tmp/ipykernel_2352966/1422480942.py:6: FutureWarning: The default value of regex will change from True to False in a future version.
  wra_rice_df['pds_total_consumption_decimal'] = wra_rice_df.pds_total_consumption_decimal.str.replace('.0', '')
/tmp/ipykernel_2352966/1422480942.py:7: FutureWarning: The default value of regex will change from True to False in a future version.
  wra_rice_df['nonpds_total_consumption_decimal'] = wra_rice_df.nonpds_total_consumption_decimal.str.replace('.0', '')


In [123]:
wra_rice_df.pds_total_consumption.describe()

count    43316.000000
mean        46.363408
std         90.318725
min          0.000000
25%          5.500000
50%         24.240000
75%         45.450000
max       2625.262500
Name: pds_total_consumption, dtype: float64

In [124]:
wra_rice_df.nonpds_total_consumption.describe()

count    132320.000000
mean        111.013397
std         294.423469
min           0.000000
25%           0.000000
50%           3.300000
75%          12.120000
max        4185.418500
Name: nonpds_total_consumption, dtype: float64

In [118]:
u5_rice_df = pd.merge(u5_df, pds_df, on='common_id', how='inner')
u5_rice_df = pd.merge(u5_rice_df, nonpds_df, on='common_id', how='inner')
u5_rice_df

,common_id,1_x,2_x,3_x,4_x,gender,age,7,8_x,9_x,...,9_y,1,2,3,4,5_y,nonpds_total_consumption_int,nonpds_total_consumption_decimal,8,9
0,HCES2022655531010131213011 201212 302,H,2,5,6,1,2,1,1,NaN,...,23213.0,F,5,102,NaN,NaN,40.000,800.0,1.0,23213.0
1,HCES2022655531010131213011 201212 302,H,2,6,6,2,4,1,1,NaN,...,23213.0,F,5,102,NaN,NaN,40.000,800.0,1.0,23213.0
2,HCES2022655531010131213011 201212 309,H,2,5,6,1,4,1,3,3.0,...,23213.0,F,5,102,NaN,NaN,10.00,200.0,1.0,23213.0
3,HCES2022655531010131213011 201212 310,H,2,9,6,1,3,1,1,NaN,...,23213.0,F,5,102,NaN,NaN,10.000,200.0,1.0,23213.0
4,HCES2022655531010131213011 201212 311,H,2,4,6,1,0,1,1,NaN,...,23213.0,F,5,102,NaN,NaN,20,400.0,1.0,23213.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
35305,HCES2022349282373710112019 201201 207,H,2,6,6,1,4,1,1,NaN,...,1272.0,F,5,102,NaN,NaN,25,800.0,1.0,1272.0
35306,HCES20223492723737101120110101201 105,H,2,4,6,1,1,1,1,NaN,...,3954.0,F,5,102,NaN,NaN,10,320.0,1.0,3954.0
35307,HCES20223492723737101120110101201 105,H,2,4,6,1,1,1,1,NaN,...,3954.0,F,5,102,NaN,NaN,10,320.0,1.0,3954.0
35308,HCES20223492023737102120110201201 203,H,2,6,6,1,2,1,1,NaN,...,848.0,F,5,102,NaN,NaN,10,400.0,1.0,848.0


In [119]:
u5_rice_df['pds_total_consumption_int'] = u5_rice_df['pds_total_consumption_int'].astype(str)
u5_rice_df['nonpds_total_consumption_int'] = u5_rice_df['nonpds_total_consumption_int'].astype(str)
u5_rice_df['pds_total_consumption_decimal'] = u5_rice_df['pds_total_consumption_decimal'].astype(str)
u5_rice_df['nonpds_total_consumption_decimal'] = u5_rice_df['nonpds_total_consumption_decimal'].astype(str)

u5_rice_df['pds_total_consumption_decimal'] = u5_rice_df.pds_total_consumption_decimal.str.replace('.0', '')
u5_rice_df['nonpds_total_consumption_decimal'] = u5_rice_df.nonpds_total_consumption_decimal.str.replace('.0', '')

u5_rice_df['pds_total_consumption'] = u5_rice_df['pds_total_consumption_decimal'] + '.' + u5_rice_df['pds_total_consumption_decimal']
u5_rice_df['nonpds_total_consumption'] = u5_rice_df['nonpds_total_consumption_decimal'] + '.' + u5_rice_df['nonpds_total_consumption_decimal']

u5_rice_df['pds_total_consumption'] = pd.to_numeric(u5_rice_df['pds_total_consumption'], errors='coerce')
u5_rice_df['nonpds_total_consumption'] = pd.to_numeric(u5_rice_df['nonpds_total_consumption'], errors='coerce')

/tmp/ipykernel_2352966/3869122219.py:6: FutureWarning: The default value of regex will change from True to False in a future version.
  u5_rice_df['pds_total_consumption_decimal'] = u5_rice_df.pds_total_consumption_decimal.str.replace('.0', '')
/tmp/ipykernel_2352966/3869122219.py:7: FutureWarning: The default value of regex will change from True to False in a future version.
  u5_rice_df['nonpds_total_consumption_decimal'] = u5_rice_df.nonpds_total_consumption_decimal.str.replace('.0', '')


In [121]:
u5_rice_df.pds_total_consumption.describe()

count    10116.000000
mean        44.510662
std         83.211363
min          0.000000
25%          5.500000
50%         24.240000
75%         45.450000
max       1875.187500
Name: pds_total_consumption, dtype: float64

In [122]:
u5_rice_df.nonpds_total_consumption.describe()

count    33114.000000
mean       102.981819
std        285.691983
min          0.000000
25%          0.000000
50%          4.400000
75%         10.100000
max       3395.339500
Name: nonpds_total_consumption, dtype: float64